# NLP — Chapitre 2 complet | spaCy Only
### EST Nador • Révision pour TD/TP et examen
**Basé sur le PDF « Chapitre 2 — Prétraitement des données textuelles » (76 pages).**

**بالدارجة:** هاد Notebook فيه الدرس من الأول للآخر بالترتيب، مع كل Fonction وLoop، وشرح، وأمثلة ونتائج كتشوفها منين كتشغل الخلايا.

**مهم:** الكود هنا فيه **spaCy وPython standard فقط**؛ ما فيه لا NLTK لا Stanza. بعض الفقرات الأصلية كتستعمل NLTK/Stanza، عطينا المفاهيم ديالها فقط. فـSection VII زدت أمثلة spaCy للتطبيق، ومعلّمين بـ **Complément** حيث الـPDF فيه نظري بالأساس.

**طريقة الاستخدام:** Google Colab → File → Upload notebook → Runtime → Run all، أو شغل الخلايا واحدة بواحدة. الموديل ضروري للتنفيذ. القيم المرتبطة بـNER/POS كتختلف أحياناً حسب نسخة الموديل.

## 0 — Installation et préparation (à exécuter une fois)
هنا كنثبتو المكتبة والموديل الإنجليزي اللي مستعمل فـPDF.

In [1]:
# Google Colab : exécuter cette cellule une fois
%pip install -q spacy emoji
!python -m spacy download en_core_web_sm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 3.4 MB/s eta 0:00:04
     -------- ------------------------------- 2.6/12.8 MB 8.2 MB/s eta 0:00:02
     ---------- ----------------------------- 3.4/12.8 MB 7.6 MB/s eta 0:00:02
     ---------- ----------------------------- 3.4/12.8 MB 7.6 MB/s eta 0:00:02
     ------------ --------------------------- 3.9/12.8 MB 4.1 MB/s eta 0:00:03
     ------------- -------------------------- 4.5/12.8 MB 3.7 MB/s eta 0:00:03
     --------------- ------------------------ 5.0/12.8 MB 3.5 MB/s eta 0:00:03
     ------------------ --------------------- 5.8/12.8 MB 3.5 MB/s eta 0:00:03
     ------------------- -------------------- 6.3/12.8 MB 3.4 MB/s eta 0:00:02
     ---------------------- ----------------- 7.1/12.8 MB 3.4 MB/s eta 0:00:02
     ---------------------- ----------------- 7.1/12.8 MB 3.4 MB/s


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import spacy
nlp = spacy.load("en_core_web_sm")
print("spaCy:", spacy.__version__)
print("Pipeline:", nlp.pipe_names)
doc = nlp("She is running. Apple was founded in 1976.")
print("Nombre de tokens:", len(doc))

spaCy: 3.8.16
Pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
Nombre de tokens: 10


## I. Tokenisation — pages 4–13
**Définition:** découper le texte en tokens (mots, nombres, ponctuation, URL…).

**الفرق:** `text` نص خام؛ `nlp(text)` كيعطينا `Doc`؛ `for token in doc` كيدوز على كل Token. `token.text` كيحتافظ بشكل الكلمة كيف جات.

In [3]:
texte = "Tesla is looking at buying U.S. startup for $6 million."
doc = nlp(texte)
for token in doc:
    print(f"{token.i:>2} | {token.text:<12} | alpha={token.is_alpha} | stop={token.is_stop}")

 0 | Tesla        | alpha=True | stop=False
 1 | is           | alpha=True | stop=True
 2 | looking      | alpha=True | stop=False
 3 | at           | alpha=True | stop=True
 4 | buying       | alpha=True | stop=False
 5 | U.S.         | alpha=False | stop=False
 6 | startup      | alpha=True | stop=False
 7 | for          | alpha=True | stop=True
 8 | $            | alpha=False | stop=False
 9 | 6            | alpha=False | stop=False
10 | million      | alpha=True | stop=False
11 | .            | alpha=False | stop=False


### Tous les attributs de base des tokens
| Propriété | شرح | Exemple |
|---|---|---|
| `token.text` | النص ديال Token | running |
| `token.i` / `token.idx` | ترتيب token / موقع الحرف الأول | 0 / 0 |
| `token.is_alpha` | حروف فقط؟ | hello: True |
| `token.is_digit` | أرقام فقط؟ | 123: True |
| `token.like_num` | شبه عدد؟ | twenty: True |
| `token.is_punct` | ترقيم؟ | !: True |
| `token.is_stop` | Stopword؟ | the: True |
| `token.is_space` | مساحة/سطر؟ | True/False |
| `token.is_upper`, `is_lower`, `is_title` | شكل الحروف | HELLO, hello, Hello |
| `token.like_url`, `like_email` | شبه رابط أو بريد | True/False |
| `token.shape_` | شكل الأحرف | Xxxx |

**مهم:** `is_alpha` و`is_punct` خصائص بلا `()`، أما `nlp(text)` فهي Function.

In [4]:
doc = nlp("Hello! I have 2 books and twenty pencils. Visit https://example.org")
for token in doc:
    print(token.text, "alpha:", token.is_alpha, "digit:", token.is_digit,
          "number:", token.like_num, "punct:", token.is_punct,
          "url:", token.like_url)
print("Liste:", [token.text for token in doc])
print("Mots seuls:", [t.text for t in doc if t.is_alpha])

Hello alpha: True digit: False number: False punct: False url: False
! alpha: False digit: False number: False punct: True url: False
I alpha: True digit: False number: False punct: False url: False
have alpha: True digit: False number: False punct: False url: False
2 alpha: False digit: True number: True punct: False url: False
books alpha: True digit: False number: False punct: False url: False
and alpha: True digit: False number: False punct: False url: False
twenty alpha: True digit: False number: True punct: False url: False
pencils alpha: True digit: False number: False punct: False url: False
. alpha: False digit: False number: False punct: True url: False
Visit alpha: True digit: False number: False punct: False url: False
https://example.org alpha: False digit: False number: False punct: False url: True
Liste: ['Hello', '!', 'I', 'have', '2', 'books', 'and', 'twenty', 'pencils', '.', 'Visit', 'https://example.org']
Mots seuls: ['Hello', 'I', 'have', 'books', 'and', 'twenty', '

### Tokenisation multilingue et limites — pages 9–13
الـPDF كيهضر على تقسيم العبارات المركبة، علامات الترقيم، URL والبريد، وكيعرض أمثلة عربية وتمارين. **spaCy tokenizer** كيعتمد على قواعد كل لغة؛ موديل الإنجليزية ماشي مخصص للدارجة. تقدر تستعمل `spacy.blank("ar")` للتقسيم الأساسي العربي، ولكن بلا تحليل POS/NER مدرب.

In [5]:
ar_nlp = spacy.blank("ar")
ar_doc = ar_nlp("أنا أحب معالجة اللغة الطبيعية، وأتعلم بايثون.")
print([token.text for token in ar_doc])
print("English tokenizer:", [t.text for t in nlp.make_doc("Email me: hello@example.com")])

['أنا', 'أحب', 'معالجة', 'اللغة', 'الطبيعية', '،', 'وأتعلم', 'بايثون', '.']
English tokenizer: ['Email', 'me', ':', 'hello@example.com']


## II. Segmentation des phrases — pages 14–29
**EOS** = End Of Sentence. `doc.sents` كتستخرج الجمل. صعوبات: `Dr.`، اختصارات، نقط داخل أرقام، `:` و`;`. كاين قواعد يدوية / apprentissage automatique / Transformers (النظرية فـPDF).

In [6]:
doc = nlp("She loves math. He prefers physics. They both enjoy science.")
for sent in doc.sents:
    print("PHRASE:", sent.text, "| start:", sent.start, "| end:", sent.end)
for token in doc:
    print(token.text, "is_sent_start:", token.is_sent_start)

PHRASE: She loves math. | start: 0 | end: 4
PHRASE: He prefers physics. | start: 4 | end: 8
PHRASE: They both enjoy science. | start: 8 | end: 13
She is_sent_start: True
loves is_sent_start: False
math is_sent_start: False
. is_sent_start: False
He is_sent_start: True
prefers is_sent_start: False
physics is_sent_start: False
. is_sent_start: False
They is_sent_start: True
both is_sent_start: False
enjoy is_sent_start: False
science is_sent_start: False
. is_sent_start: False


### Composant personnalisé `@Language.component` — Exercice p.26–27
نديرو `:` كحد ديال جملة فقط إلا تبعاتها مساحة وحرف كبير. خاص **نضيفو هاد المكوّن قبل `parser`**، باش نعلمو بدايات الجمل. `token.whitespace_` فيه المسافة اللي جات من بعد token. المثال مرتبط بالقواعد اللي فـPDF.

In [7]:
from spacy.language import Language

@Language.component("colon_sentence_boundary_ch2")
def colon_sentence_boundary_ch2(doc):
    for i, token in enumerate(doc[:-1]):
        suivant = doc[i + 1]
        if (token.text == ":" and token.whitespace_ == " "
                and suivant.text and suivant.text[0].isupper()):
            suivant.is_sent_start = True
    return doc

nlp_seg = spacy.load("en_core_web_sm")
nlp_seg.add_pipe("colon_sentence_boundary_ch2", before="parser")
doc_seg = nlp_seg("She loves math: He prefers physics. They both enjoy science.")
for sent in doc_seg.sents:
    print(sent.text)

She loves math:
He prefers physics.
They both enjoy science.


### Segmentation `;` — Complément direct du même principe
إلى بغيتي `;` حد ديال جملة، كتبدل شرط `token.text`. ما تديرش علامة الترقيم دائماً نهاية جملة، خاصك القاعدة المناسبة للتطبيق.

In [8]:
doc = nlp("Hello world. Dr. Smith works here. Another sentence!")
print([sent.text for sent in doc.sents])

['Hello world.', 'Dr. Smith works here.', 'Another sentence!']


## POS Tagging — pages 30–36 (numérotation مكررة في الـPDF)
`token.pos_` **catégorie grammaticale générale** (VERB, NOUN, PROPN, ADJ…). `token.tag_` **détaillée** (VBD, VBZ, NNS…). `spacy.explain()` كتعطي التفسير. 

In [9]:
doc = nlp("She loves cats and was running quickly.")
for token in doc:
    print(f"{token.text:<12} {token.pos_:<8} {token.tag_:<6} {spacy.explain(token.tag_)}")
print("VBZ:", spacy.explain("VBZ"))

She          PRON     PRP    pronoun, personal
loves        VERB     VBZ    verb, 3rd person singular present
cats         NOUN     NNS    noun, plural
and          CCONJ    CC     conjunction, coordinating
was          AUX      VBD    verb, past tense
running      VERB     VBG    verb, gerund or present participle
quickly      ADV      RB     adverb
.            PUNCT    .      punctuation mark, sentence closer
VBZ: verb, 3rd person singular present


### POS courants
| POS | Signification |
|---|---|
| NOUN | Nom commun |
| PROPN | Nom propre |
| VERB | Verbe |
| AUX | Auxiliaire |
| PRON | Pronom |
| DET | Déterminant |
| ADJ | Adjectif |
| ADV | Adverbe |
| ADP | Préposition |
| CCONJ / SCONJ | Coordination / subordination |
| NUM | Nombre |
| PUNCT | Ponctuation |

**Statistiques :** le cours utilise `doc.count_by(spacy.attrs.POS)` pour compter les étiquettes.

In [10]:
from collections import Counter
print("Counter POS:", Counter(token.pos_ for token in doc))
counts = doc.count_by(spacy.attrs.POS)
for pos_id, combien in counts.items():
    print(doc.vocab[pos_id].text, combien)

Counter POS: Counter({'VERB': 2, 'PRON': 1, 'NOUN': 1, 'CCONJ': 1, 'AUX': 1, 'ADV': 1, 'PUNCT': 1})
PRON 1
VERB 2
NOUN 1
CCONJ 1
AUX 1
ADV 1
PUNCT 1


## III. Normalisation — pages 37–58
### III.1 Normalisation lexicale (pages 37–47)
1. Case folding (`lower()`)
2. Suppression ponctuation (`string.punctuation`, `str.maketrans`, `translate`)
3. Abréviations / formes canoniques (`replace`)
4. Nombres et symboles (`re.sub`)
5. Emojis: supprimer par Unicode regex ou convertir avec `emoji.demojize`

**Attention:** Supprimer `not`, signes ou emojis peut perdre le sens; choisir selon tâche NLP. 

In [11]:
import string
texte = "HELLO, World! NLP in 2026."
print("1 Lower:", texte.lower())
translator = str.maketrans("", "", string.punctuation)
print("2 Sans ponctuation:", texte.lower().translate(translator))
print("3 Sans ponctuation ni chiffres:", texte.lower().translate(
    str.maketrans("", "", string.punctuation + string.digits)))

1 Lower: hello, world! nlp in 2026.
2 Sans ponctuation: hello world nlp in 2026
3 Sans ponctuation ni chiffres: hello world nlp in 


In [12]:
abbreviations = {"U.S.A.": "USA", "Dr.": "Doctor", "w/": "with", "&": "and"}
texte = "Dr. Smith works in the U.S.A. w/ Bob & Alice."
for abbr, forme in abbreviations.items():
    texte = texte.replace(abbr, forme)
print(texte)
# Attention: replace() est littéral, il peut toucher des sous-chaînes; les frontières de mot exigent plus de soin.

Doctor Smith works in the USA with Bob and Alice.


In [13]:
import re
texte = "I paid $250 in 2026 for 2 books."
print("Chiffres masqués:", re.sub(r"\d+(?:[.,]\d+)*", "<NUM>", texte))
print("Chiffres supprimés:", re.sub(r"\d+", "", texte))
print("Espaces uniformisés:", re.sub(r"\s+", " ", texte).strip())
# Les masquages <NUM> sont des chaînes de texte, pas des tokens spéciaux spaCy par défaut.

Chiffres masqués: I paid $<NUM> in <NUM> for <NUM> books.
Chiffres supprimés: I paid $ in  for  books.
Espaces uniformisés: I paid $250 in 2026 for 2 books.


In [14]:
# Unicode regex — exemple de nettoyage des emoji fréquents du cours (pas universel)
emoji_pattern = re.compile(
    "[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]+"
)
def remove_emojis(text):
    return emoji_pattern.sub("", text)
exemples = ["Hello 😊!", "Super 🚀 projet!", "Texte simple."]
for t in exemples:
    print("AVANT:", t, "| APRES:", remove_emojis(t))

import emoji
print("Convertir:", emoji.demojize("I love Python ❤️", delimiters=("[", "]")))
# Complément de prudence: les plages Unicode ci-dessus ne capturent pas tous les emoji composés.

AVANT: Hello 😊! | APRES: Hello !
AVANT: Super 🚀 projet! | APRES: Super  projet!
AVANT: Texte simple. | APRES: Texte simple.
Convertir: I love Python [red_heart]


### III.2 Normalisation morphologique — pages 48–58
**Stemming** = raccourcir selon des règles, peut produire une racine non lexicale. **Lemmatisation** = forme canonique selon POS et contexte. Le PDF donne **Stemming avec NLTK**: pas de code NLTK dans ce notebook.

Exemple: `running → run`, `was → be`, `cats → cat`. `token.lemma_` est la propriété spaCy pour le lemme.

In [15]:
doc = nlp("I am meeting him at the meeting. He was running and loves cats.")
for token in doc:
    if token.is_alpha:
        print(f"{token.text:<12} | {token.pos_:<7} | {token.lemma_}")
lemmas = [t.lemma_ for t in doc if t.is_alpha]
print("Lemmes:", lemmas)

I            | PRON    | I
am           | AUX     | be
meeting      | VERB    | meet
him          | PRON    | he
at           | ADP     | at
the          | DET     | the
meeting      | NOUN    | meeting
He           | PRON    | he
was          | AUX     | be
running      | VERB    | run
and          | CCONJ   | and
loves        | VERB    | love
cats         | NOUN    | cat
Lemmes: ['I', 'be', 'meet', 'he', 'at', 'the', 'meeting', 'he', 'be', 'run', 'and', 'love', 'cat']


## IV. NER — Reconnaissance d'entités nommées — pages 59–62
NER détecte personnes (`PERSON`), organisations (`ORG`), villes/pays (`GPE`), dates (`DATE`), montants (`MONEY`). `doc.ents` → entités, `ent.text` → texte, `ent.label_` → catégorie. Les prédictions dépendent du modèle.

In [16]:
texte = "Apple was founded by Steve Jobs in Cupertino in 1976."
doc = nlp(texte)
for ent in doc.ents:
    print(ent.text, "|", ent.label_, "|", spacy.explain(ent.label_))
print("Types:", Counter(ent.label_ for ent in doc.ents))
print("ORG/PERSON:", [(e.text,e.label_) for e in doc.ents if e.label_ in ["ORG", "PERSON"]])

Apple | ORG | Companies, agencies, institutions, etc.
Steve Jobs | PERSON | People, including fictional
Cupertino | GPE | Countries, cities, states
1976 | DATE | Absolute or relative dates or periods
Types: Counter({'ORG': 1, 'PERSON': 1, 'GPE': 1, 'DATE': 1})
ORG/PERSON: [('Apple', 'ORG'), ('Steve Jobs', 'PERSON')]


## V. Stopwords — pages 63–66
`token.is_stop` = stopword ? `not token.is_stop` = garder mots non-stopwords. Pour filtrer la ponctuation : `token.is_alpha`. **Important:** pour ajouter un stopword, actualiser à la fois `nlp.Defaults.stop_words` et `nlp.vocab[mot].is_stop`. Le PDF ajoute un mot comme `very`.

In [17]:
nlp_stop = spacy.load("en_core_web_sm")
texte = "The cat is sleeping on the mat because it is tired and very hungry."
doc = nlp_stop(texte)
print("Stopwords texte:", [t.text for t in doc if t.is_stop])
print("Mots utiles:", [t.text for t in doc if t.is_alpha and not t.is_stop])
print("Nombre stopwords du modèle:", len(nlp_stop.Defaults.stop_words))

nlp_stop.Defaults.stop_words.add("hungry")
nlp_stop.vocab["hungry"].is_stop = True
print("Après ajout hungry:", [t.text for t in nlp_stop(texte) if t.is_alpha and not t.is_stop])

Stopwords texte: ['The', 'is', 'on', 'the', 'because', 'it', 'is', 'and', 'very']
Mots utiles: ['cat', 'sleeping', 'mat', 'tired', 'hungry']
Nombre stopwords du modèle: 326
Après ajout hungry: ['cat', 'sleeping', 'mat', 'tired']


## VI. Les Matchers — pages 67–72
**Pattern** = الشروط. **Matcher** = الأداة اللي كتبحث. **Match** = النتيجة. 

- `Matcher` : recherche selon attributs de tokens (`LOWER`, `LEMMA`, `POS`, `TEXT`, `LIKE_NUM`).
- `PhraseMatcher` : liste de mots/expressions déjà connus.
- `DependencyMatcher` : motifs liés aux dépendances syntaxiques (**présent en théorie p.68**, exemple pratique complément ci-dessous).

**فرق مهم:** `token.lemma_` كيعطي الـlemme، أما `{"LEMMA": "buy"}` شرط في pattern.

In [18]:
from spacy.matcher import Matcher
matcher = Matcher(nlp.vocab)
matcher.add("PRODUCT", [[{"LOWER": "iphone"}], [{"LOWER": "ipad"}]])
doc = nlp("I bought an iPhone and my friend has an iPad.")
for match_id, start, end in matcher(doc):
    print(nlp.vocab.strings[match_id], "=>", doc[start:end].text,
          "indices:", start, end)

PRODUCT => iPhone indices: 3 4
PRODUCT => iPad indices: 9 10


In [19]:
# Matcher complexe du chapitre: buy + an + PROPN
matcher = Matcher(nlp.vocab)
pattern = [
    {"LEMMA": "buy"},
    {"LOWER": "an"},
    {"POS": "PROPN"}
]
matcher.add("ACHAT_GADGET", [pattern])
doc = nlp("Yesterday I bought an iPhone. My friend bought a Tesla.")
for match_id, start, end in matcher(doc):
    print("Motif trouvé:", doc[start:end].text)

Motif trouvé: bought an iPhone


In [20]:
from spacy.matcher import PhraseMatcher
phrase_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
phrases = [nlp.make_doc(p) for p in ["iPhone", "iPad", "MacBook Pro"]]
phrase_matcher.add("PRODUITS", phrases)
doc = nlp("I want to buy an iPhone or a MacBook Pro.")
for match_id, start, end in phrase_matcher(doc):
    print("Produit trouvé:", doc[start:end].text)

Produit trouvé: iPhone
Produit trouvé: MacBook Pro


### Exemple de Matcher arabe — p.72
Le cours applique un Matcher à plusieurs variantes du mot arabe `كتب`. Ici **spaCy.blank("ar")** sert uniquement au tokenizer et au Matcher avec `TEXT` (sans morphologie entraînée).

In [21]:
nlp_ar = spacy.blank("ar")
matcher_ar = Matcher(nlp_ar.vocab)
formes = ["كتب", "كتبت", "كتابة", "يكتب", "الكتاب"]
matcher_ar.add("KATABA", [[{"TEXT": {"IN": formes}}]])
doc_ar = nlp_ar("كتب علي الدرس. يكتب التلميذ كتابة جميلة.")
for match_id, start, end in matcher_ar(doc_ar):
    print("Mot arabe:", doc_ar[start:end].text)

Mot arabe: كتب
Mot arabe: يكتب
Mot arabe: كتابة


## VII. La structure syntaxique — pages 73–76
**Deux approches du PDF:**
- **Constituency Parsing:** arbre des groupes `S`, `NP`, `VP`, `Det`, `PP`, `ADJP`, `ADVP`.
- **Dependency Parsing:** pour chaque token, une relation vers sa tête (`ROOT`, `nsubj`, `dobj`, `det` selon les conventions du cours).

**Exemple du PDF:** « Marie mange une pomme ». `mange` racine, `Marie` sujet de `mange`, `pomme` objet de `mange`, `une` déterminant de `pomme`.

**Complément spaCy (pas un code donné dans les pages 73–76):** `token.dep_`, `token.head.text`, `token.children` pour explorer les dépendances. Les étiquettes exactes de spaCy peuvent différer selon langue et modèle.

In [22]:
# Complément pratique: dépendances anglaises avec modèle entraîné
analyse = nlp("Marie eats an apple.")
for token in analyse:
    print(f"{token.text:<10} POS={token.pos_:<6} DEP={token.dep_:<8} HEAD={token.head.text}")
print("Noun chunks:", [chunk.text for chunk in analyse.noun_chunks])
# Il n'y a pas de constituency parser natif dans cette démo spaCy.

Marie      POS=PROPN  DEP=nsubj    HEAD=eats
eats       POS=VERB   DEP=ROOT     HEAD=eats
an         POS=DET    DEP=det      HEAD=apple
apple      POS=NOUN   DEP=dobj     HEAD=eats
.          POS=PUNCT  DEP=punct    HEAD=eats
Noun chunks: ['Marie', 'an apple']


## Fiche de révision — Toutes les boucles importantes
| Objectif | Boucle / code | Résultat |
|---|---|---|
| Tokenisation | `for token in doc` | `token.text` |
| POS | `for token in doc` | `token.pos_`, `token.tag_` |
| Lemmatisation | `for token in doc` | `token.lemma_` |
| Segmentation | `for sent in doc.sents` | `sent.text` |
| NER | `for ent in doc.ents` | `ent.text`, `ent.label_` |
| Stopwords | `if token.is_stop` | filtre |
| Matcher | `for match_id, start, end in matcher(doc)` | `doc[start:end]` |
| PhraseMatcher | `for match_id, start, end in phrase_matcher(doc)` | `span.text` |
| Dépendances | `for token in doc` | `token.dep_`, `token.head.text` |

**Repères techniques:** `LOWER` (pattern) ≠ `str.lower()` (Python); `LEMMA` (pattern) ≠ `token.lemma_` (attribut); `is_alpha` (booléen) ≠ `is_stop` (booléen).

## TP intégré — Un seul code avec toutes les principales étapes spaCy
**Application combinée supplémentaire de révision**, pas un exercice identique à une page précise : texte → segmentation → tokens → POS/lemma → NER → stopwords → Matcher → syntactic dependencies.

In [23]:
import spacy
from spacy.matcher import Matcher
from collections import Counter

nlp_full = spacy.load("en_core_web_sm")
text = "Apple launched the iPhone in 2007. Steve Jobs loved this product!"
doc = nlp_full(text)

print("1 PHRASES")
for sent in doc.sents: print(sent.text)

print("\n2 TOKENS + POS + LEMMA")
for t in doc: print(t.text, t.pos_, t.lemma_, "alpha:", t.is_alpha)

print("\n3 NER")
for ent in doc.ents: print(ent.text, ent.label_)

print("\n4 SANS STOPWORDS")
print([t.text for t in doc if t.is_alpha and not t.is_stop])

print("\n5 MATCHER")
matcher_full = Matcher(nlp_full.vocab)
matcher_full.add("PRODUIT", [[{"LOWER": "iphone"}]])
for match_id, start, end in matcher_full(doc):
    print(nlp_full.vocab.strings[match_id], doc[start:end].text)

print("\n6 DEPENDANCES")
for t in doc: print(t.text, "=>", t.head.text, "[", t.dep_, "]")

1 PHRASES
Apple launched the iPhone in 2007.
Steve Jobs loved this product!

2 TOKENS + POS + LEMMA
Apple PROPN Apple alpha: True
launched VERB launch alpha: True
the DET the alpha: True
iPhone PROPN iPhone alpha: True
in ADP in alpha: True
2007 NUM 2007 alpha: False
. PUNCT . alpha: False
Steve PROPN Steve alpha: True
Jobs PROPN Jobs alpha: True
loved VERB love alpha: True
this DET this alpha: True
product NOUN product alpha: True
! PUNCT ! alpha: False

3 NER
Apple ORG
iPhone ORG
2007 DATE
Steve Jobs PERSON

4 SANS STOPWORDS
['Apple', 'launched', 'iPhone', 'Steve', 'Jobs', 'loved', 'product']

5 MATCHER
PRODUIT iPhone

6 DEPENDANCES
Apple => launched [ nsubj ]
launched => launched [ ROOT ]
the => iPhone [ det ]
iPhone => launched [ dobj ]
in => launched [ prep ]
2007 => in [ pobj ]
. => launched [ punct ]
Steve => Jobs [ compound ]
Jobs => loved [ nsubj ]
loved => loved [ ROOT ]
this => product [ det ]
product => loved [ dobj ]
! => loved [ punct ]


## Questions d’examen (entraînement autonome)
1. Quelle différence entre `doc.sents`, `doc.ents` et `for token in doc` ?
2. Quelle différence entre `token.pos_`, `token.tag_` et `token.lemma_` ?
3. Pourquoi `token.is_alpha` ne supprime-t-il pas les stopwords ?
4. Que signifie `OP: "?"` dans un Matcher ?
5. Pourquoi une liste `PhraseMatcher` ne remplace-t-elle pas totalement NER ?
6. Quelle différence entre Stemming et Lemmatisation ?
7. Expliquez `doc[start:end]` et `nlp.vocab.strings[match_id]`.
8. Montrez la différence entre normalisation lexicale et morphologique.
9. Quelle différence entre constituency et dependency parsing ?
10. Pourquoi le nettoyage peut-il supprimer des informations utiles (`not`, emoji, ponctuation) ?

✅ **Fin du Chapitre 2.** Toutes les parties du support I–VII sont reprises; les implémentations NLTK/Stanza sont exclues à ta demande.